# 📊 Módulo 6: Métodos Estatísticos e Mineração Não-Supervisionada



## 🎯 Objetivos
- Aplicar testes de hipótese para comparar classificadores
- Implementar K-Means do zero e explorar variantes
- Comparar algoritmos de clustering (hierárquico, DBSCAN, HDBSCAN, GMM)
- Minerar regras de associação com mlxtend
- Detectar anomalias com múltiplos métodos

## 📊 Sumário
1. Fundamentos Estatísticos — Testes de Hipótese
2. K-Means: Conceito e Implementação
3. Clustering Hierárquico
4. DBSCAN e HDBSCAN
5. Modelos de Mistura Gaussiana (GMM)
6. Avaliação de Clustering
7. Regras de Associação
8. Detecção de Anomalias
9. Exercícios

In [ ]:
# Instalação das dependências
!pip install -q scikit-learn pandas numpy matplotlib seaborn scipy plotly mlxtend
!pip install -q hdbscan pyod networkx

## 📦 Setup e Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn — clustering
from sklearn.datasets import (make_blobs, make_moons, make_circles,
                               load_iris, load_wine, load_breast_cancer)
from sklearn.cluster import (KMeans, AgglomerativeClustering,
                              DBSCAN, SpectralClustering)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score, silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score,
    normalized_mutual_info_score, roc_auc_score
)
import xgboost as xgb

# Scipy
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage

# Mlxtend — regras de associação
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# HDBSCAN
try:
    import hdbscan
    HDBSCAN_AVAIL = True
except ImportError:
    HDBSCAN_AVAIL = False
    print('hdbscan não instalado. Usando sklearn.cluster.HDBSCAN se disponível.')
    try:
        from sklearn.cluster import HDBSCAN
        HDBSCAN_AVAIL = True
    except ImportError:
        HDBSCAN_AVAIL = False

# PyOD
try:
    from pyod.models.iforest import IForest
    from pyod.models.lof import LOF
    from pyod.models.ocsvm import OCSVM
    from pyod.models.knn import KNN as KNN_OD
    PYOD_AVAIL = True
except ImportError:
    PYOD_AVAIL = False
    print('PyOD nao disponivel.')

# Networkx para grafos de regras
try:
    import networkx as nx
    NX_AVAIL = True
except ImportError:
    NX_AVAIL = False

SEED = 42
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#005AB4', '#E67300', '#228B22', '#B4142A', '#6A0DAD', '#008B8B']

print('Setup concluído!')
print(f'HDBSCAN disponível: {HDBSCAN_AVAIL}')
print(f'PyOD disponível: {PYOD_AVAIL}')

## 📈 1. Fundamentos Estatísticos — Testes de Hipótese

Comparando classificadores com testes estatísticos rigorosos.

In [ ]:
# ================================================================
# Teste de Wilcoxon e McNemar para comparar classificadores
# ================================================================

from scipy.stats import wilcoxon, ttest_rel
from sklearn.model_selection import StratifiedKFold

# Dataset
data = load_breast_cancer()
Xh, yh = StandardScaler().fit_transform(data.data), data.target

# Modelos a comparar
clf_rf  = RandomForestClassifier(n_estimators=100, random_state=SEED)
clf_xgb = xgb.XGBClassifier(n_estimators=100, random_state=SEED,
                              eval_metric='logloss', use_label_encoder=False)

# Validação cruzada estratificada 10-fold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

scores_rf  = []
scores_xgb = []
preds_rf_all  = np.zeros(len(yh))
preds_xgb_all = np.zeros(len(yh))

for train_idx, test_idx in cv.split(Xh, yh):
    X_tr, X_te = Xh[train_idx], Xh[test_idx]
    y_tr, y_te = yh[train_idx], yh[test_idx]

    clf_rf.fit(X_tr, y_tr)
    clf_xgb.fit(X_tr, y_tr)

    p_rf  = clf_rf.predict(X_te)
    p_xgb = clf_xgb.predict(X_te)

    scores_rf.append(accuracy_score(y_te, p_rf))
    scores_xgb.append(accuracy_score(y_te, p_xgb))

    preds_rf_all[test_idx]  = p_rf
    preds_xgb_all[test_idx] = p_xgb

scores_rf  = np.array(scores_rf)
scores_xgb = np.array(scores_xgb)

print('='*55)
print('COMPARAÇÃO DE CLASSIFICADORES — BREAST CANCER')
print('='*55)
print(f'Random Forest : {scores_rf.mean():.4f} ± {scores_rf.std():.4f}')
print(f'XGBoost       : {scores_xgb.mean():.4f} ± {scores_xgb.std():.4f}')

# Teste t pareado
t_stat, p_ttest = ttest_rel(scores_rf, scores_xgb)
print(f'\nTeste t pareado   : t={t_stat:.4f}, p={p_ttest:.4f}')

# Teste de Wilcoxon signed-rank
try:
    w_stat, p_wilcoxon = wilcoxon(scores_rf, scores_xgb)
    print(f'Wilcoxon signed-rank: W={w_stat:.1f}, p={p_wilcoxon:.4f}')
except Exception as e:
    p_wilcoxon = 1.0
    print(f'Wilcoxon: {e}')

alpha = 0.05
print(f'\nNível de significância α = {alpha}')
if p_wilcoxon < alpha:
    print(f'→ REJEITA H0: há diferença significativa entre os classificadores (p={p_wilcoxon:.4f})')
else:
    print(f'→ NÃO rejeita H0: diferença não é significativa (p={p_wilcoxon:.4f})')

# Teste de McNemar
b = np.sum((preds_rf_all == yh) & (preds_xgb_all != yh))  # RF acerta, XGB erra
c = np.sum((preds_rf_all != yh) & (preds_xgb_all == yh))  # XGB acerta, RF erra
mcnemar_stat = (abs(b - c) - 1)**2 / (b + c) if (b + c) > 0 else 0
p_mcnemar = 1 - stats.chi2.cdf(mcnemar_stat, df=1)
print(f'\nMcNemar: b={b}, c={c}, χ²={mcnemar_stat:.4f}, p={p_mcnemar:.4f}')

# Boxplot das acurácias
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot([scores_rf, scores_xgb], labels=['Random Forest', 'XGBoost'],
                patch_artist=True,
                boxprops=dict(facecolor=COLORS[0], alpha=0.6))
axes[0].set_ylabel('Acurácia (10-fold CV)')
axes[0].set_title('Acurácia por Fold', fontweight='bold')
axes[0].axhline(scores_rf.mean(), color=COLORS[0], linestyle='--', alpha=0.5)
axes[0].axhline(scores_xgb.mean(), color=COLORS[1], linestyle='--', alpha=0.5)

x_folds = np.arange(1, 11)
axes[1].plot(x_folds, scores_rf, 'o-', color=COLORS[0], label='Random Forest', linewidth=2)
axes[1].plot(x_folds, scores_xgb, 's--', color=COLORS[1], label='XGBoost', linewidth=2)
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('Acurácia')
axes[1].set_title('Acurácia por Fold (Comparativo)', fontweight='bold')
axes[1].legend()
plt.suptitle('Teste de Hipótese — Comparação de Classificadores', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🎯 2. K-Means: Conceito e Implementação

In [ ]:
# ================================================================
# K-Means do Zero com NumPy — Visualização Passo a Passo
# ================================================================

class KMeansNumPy:
    """Implementação do K-Means (Lloyd's algorithm) com NumPy puro."""

    def __init__(self, K, max_iter=100, tol=1e-6, random_state=42):
        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.rng = np.random.RandomState(random_state)
        self.history = []  # salva (centroides, labels) a cada iteração

    def _assign(self, X):
        """Atribui cada ponto ao centróide mais próximo (passo E)."""
        # Distâncias: (N, K)
        dists = np.linalg.norm(X[:, np.newaxis] - self.centroids[np.newaxis, :], axis=2)
        return np.argmin(dists, axis=1)

    def _update(self, X, labels):
        """Recalcula centróides (passo M)."""
        new_centroids = np.array([
            X[labels == k].mean(axis=0) if np.sum(labels == k) > 0 else self.centroids[k]
            for k in range(self.K)
        ])
        return new_centroids

    def fit(self, X):
        N, D = X.shape
        # Inicialização aleatória
        idx = self.rng.choice(N, self.K, replace=False)
        self.centroids = X[idx].copy()

        for it in range(self.max_iter):
            labels = self._assign(X)
            self.history.append((self.centroids.copy(), labels.copy()))
            new_centroids = self._update(X, labels)
            # Critério de parada
            if np.linalg.norm(new_centroids - self.centroids) < self.tol:
                self.centroids = new_centroids
                break
            self.centroids = new_centroids

        self.labels_ = self._assign(X)
        self.inertia_ = sum(
            np.sum((X[self.labels_ == k] - self.centroids[k])**2)
            for k in range(self.K)
        )
        self.n_iter_ = len(self.history)
        return self

# Gera dataset com 3 clusters
X_km, y_km = make_blobs(n_samples=300, centers=3, cluster_std=1.2, random_state=SEED)

# Treina K-Means
km = KMeansNumPy(K=3, random_state=SEED)
km.fit(X_km)
print(f'K-Means convergiu em {km.n_iter_} iterações | Inércia: {km.inertia_:.2f}')

# Visualiza passos
steps_to_show = [0, 1, 2, km.n_iter_ - 1]
titles = ['Inicialização', 'Iteração 2', 'Iteração 3', 'Convergência']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, step, title in zip(axes, steps_to_show, titles):
    if step >= len(km.history):
        step = len(km.history) - 1
    centroids_s, labels_s = km.history[step]
    for k, c in enumerate(COLORS[:3]):
        mask = labels_s == k
        ax.scatter(X_km[mask, 0], X_km[mask, 1], c=c, alpha=0.5, s=20)
        ax.scatter(centroids_s[k, 0], centroids_s[k, 1], c=c,
                   marker='*', s=250, edgecolors='black', linewidth=1.2, zorder=5)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Convergência do K-Means — Passo a Passo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Seleção do K Ótimo: Método do Cotovelo + Silhouette
# ================================================================

K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km_sk = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    labels_k = km_sk.fit_predict(X_km)
    inertias.append(km_sk.inertia_)
    silhouettes.append(silhouette_score(X_km, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cotovelo
axes[0].plot(K_range, inertias, 'o-', color=COLORS[0], linewidth=2.5, markersize=6)
axes[0].set_xlabel('Número de Clusters K'); axes[0].set_ylabel('Inércia (WCSS)')
axes[0].set_title('Método do Cotovelo', fontweight='bold')
axes[0].axvline(3, color='red', linestyle='--', alpha=0.6, label='K ótimo')
axes[0].legend()

# Silhouette
axes[1].plot(K_range, silhouettes, 's-', color=COLORS[1], linewidth=2.5, markersize=6)
axes[1].set_xlabel('Número de Clusters K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Análise de Silhouette', fontweight='bold')
best_k = list(K_range)[np.argmax(silhouettes)]
axes[1].axvline(best_k, color='red', linestyle='--', alpha=0.6, label=f'K ótimo={best_k}')
axes[1].legend()

plt.suptitle('Escolha do K Ótimo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'K com maior Silhouette: {best_k} (score={max(silhouettes):.4f})')

# Resultado final com K ótimo
km_final = KMeans(n_clusters=best_k, n_init=10, random_state=SEED)
labels_final = km_final.fit_predict(X_km)

plt.figure(figsize=(6, 5))
for k, c in enumerate(COLORS[:best_k]):
    mask = labels_final == k
    plt.scatter(X_km[mask, 0], X_km[mask, 1], c=c, alpha=0.6, s=30, label=f'Cluster {k+1}')
    plt.scatter(km_final.cluster_centers_[k, 0], km_final.cluster_centers_[k, 1],
                c=c, marker='*', s=300, edgecolors='black', linewidth=1.5, zorder=5)
plt.title(f'K-Means com K={best_k} (resultado final)', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

## 🌳 3. Clustering Hierárquico

In [ ]:
# ================================================================
# Dendrogramas com os 4 métodos de linkage
# ================================================================

from scipy.cluster.hierarchy import dendrogram, linkage, cophenet
from scipy.spatial.distance import pdist

# Usa subconjunto do Iris para visualização clara
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
y_iris = iris.target

# Subconjunto de 50 amostras para dendrograma legível
idx_sub = np.random.choice(len(X_iris), 50, replace=False)
X_sub = X_iris[idx_sub]

linkage_methods = ['single', 'complete', 'average', 'ward']
linkage_names   = ['Single Linkage\n(vizinho mais próximo)',
                   'Complete Linkage\n(vizinho mais distante)',
                   'Average Linkage\n(UPGMA)',
                   'Ward\n(mínima variância)']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, method, name in zip(axes, linkage_methods, linkage_names):
    Z = linkage(X_sub, method=method)
    coph_corr, _ = cophenet(Z, pdist(X_sub))
    dendrogram(Z, ax=ax, leaf_rotation=90, leaf_font_size=6,
               color_threshold=0.6 * max(Z[:, 2]),
               above_threshold_color='gray')
    ax.set_title(f'{name}\n(Corr. Cofenética: {coph_corr:.3f})', fontweight='bold', fontsize=10)
    ax.set_xlabel('Índice da Amostra', fontsize=8)
    ax.set_ylabel('Distância', fontsize=8)

plt.suptitle('Dendrogramas — Clustering Hierárquico Aglomerativo (Iris, n=50)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Clustering Hierárquico vs Classes Reais do Iris
# ================================================================

linkage_sk = ['single', 'complete', 'average', 'ward']
hclust_results = []

for method in linkage_sk:
    agg = AgglomerativeClustering(n_clusters=3, linkage=method)
    preds = agg.fit_predict(X_iris)
    ari = adjusted_rand_score(y_iris, preds)
    nmi = normalized_mutual_info_score(y_iris, preds)
    sil = silhouette_score(X_iris, preds)
    hclust_results.append({'Linkage': method.capitalize(), 'ARI': round(ari, 4),
                            'NMI': round(nmi, 4), 'Silhouette': round(sil, 4)})
    print(f'{method.capitalize():10s}: ARI={ari:.4f} | NMI={nmi:.4f} | Silhouette={sil:.4f}')

df_hclust = pd.DataFrame(hclust_results)
print('\n--- Tabela: Hierarchical Clustering no Iris ---')
print(df_hclust.to_string(index=False))

# Visualização dos clusters (PCA 2D)
from sklearn.decomposition import PCA
X_iris_2d = PCA(n_components=2).fit_transform(X_iris)

fig, axes = plt.subplots(1, 5, figsize=(17, 3.5))
# Ground truth
for k, c in enumerate(COLORS[:3]):
    mask = y_iris == k
    axes[0].scatter(X_iris_2d[mask, 0], X_iris_2d[mask, 1], c=c, s=15, alpha=0.7)
axes[0].set_title('Ground Truth', fontweight='bold', fontsize=9)

for i, method in enumerate(linkage_sk):
    agg = AgglomerativeClustering(n_clusters=3, linkage=method)
    preds = agg.fit_predict(X_iris)
    for k, c in enumerate(COLORS[:3]):
        mask = preds == k
        axes[i+1].scatter(X_iris_2d[mask, 0], X_iris_2d[mask, 1], c=c, s=15, alpha=0.7)
    axes[i+1].set_title(f'{method.capitalize()}\nARI={adjusted_rand_score(y_iris,preds):.3f}',
                         fontweight='bold', fontsize=9)

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Clustering Hierárquico — Comparação de Linkages (Iris, PCA 2D)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔵 4. DBSCAN e HDBSCAN

In [ ]:
# ================================================================
# DBSCAN vs K-Means em datasets não-convexos
# ================================================================

# Gera datasets desafiadores
datasets = [
    make_moons(n_samples=300, noise=0.08, random_state=SEED),
    make_circles(n_samples=300, noise=0.05, factor=0.4, random_state=SEED),
    make_blobs(n_samples=300, centers=3, cluster_std=0.9, random_state=SEED),
]

eps_list = [0.25, 0.2, 0.6]
minpts_list = [5, 5, 5]

# Configurações de HDBSCAN
def get_hdbscan(min_cluster_size=10):
    if HDBSCAN_AVAIL:
        try:
            return hdbscan.HDBSCAN(min_cluster_size=min_cluster_size)
        except:
            from sklearn.cluster import HDBSCAN
            return HDBSCAN(min_cluster_size=min_cluster_size)
    return None

fig, axes = plt.subplots(3, 3, figsize=(13, 11))
method_titles = ['K-Means (K=2)', 'DBSCAN', 'HDBSCAN']

for row, ((X_d, y_d), ds_name, eps, minpts) in enumerate(zip(datasets, ['Moons','Circles','Blobs'], eps_list, minpts_list)):
    X_sc = StandardScaler().fit_transform(X_d)

    # K-Means
    km_d = KMeans(n_clusters=2, random_state=SEED, n_init=10)
    lbl_km = km_d.fit_predict(X_sc)

    # DBSCAN
    db = DBSCAN(eps=eps, min_samples=minpts)
    lbl_db = db.fit_predict(X_sc)

    # HDBSCAN
    hdb_clf = get_hdbscan(min_cluster_size=15)
    if hdb_clf is not None:
        lbl_hdb = hdb_clf.fit_predict(X_sc)
    else:
        lbl_hdb = lbl_db

    preds_list = [lbl_km, lbl_db, lbl_hdb]

    for col, (lbls, title) in enumerate(zip(preds_list, method_titles)):
        ax = axes[row][col]
        unique_labels = set(lbls)
        palette = {l: COLORS[i % len(COLORS)] for i, l in enumerate(sorted(unique_labels)) if l >= 0}
        palette[-1] = '#AAAAAA'  # ruído

        for label in unique_labels:
            mask = lbls == label
            style = {'s': 8, 'alpha': 0.7 if label >= 0 else 0.3,
                     'marker': 'o' if label >= 0 else 'x'}
            ax.scatter(X_sc[mask, 0], X_sc[mask, 1], c=palette.get(label, 'gray'), **style)

        n_clusters = len(set(lbls)) - (1 if -1 in lbls else 0)
        n_noise = np.sum(lbls == -1)
        info = f'K={n_clusters}'
        if n_noise > 0: info += f', ruído={n_noise}'
        ax.set_title(f'{title}\n{info}', fontsize=9, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(ds_name, fontsize=11, fontweight='bold', labelpad=5)

plt.suptitle('K-Means vs DBSCAN vs HDBSCAN em Datasets Não-Convexos',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Sensibilidade dos Parâmetros do DBSCAN
# ================================================================

X_sens, _ = make_moons(n_samples=200, noise=0.08, random_state=SEED)
X_sens = StandardScaler().fit_transform(X_sens)

eps_values = [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
min_samples_values = [2, 5, 10, 15]

# Heatmap de número de clusters
heatmap_clusters = np.zeros((len(min_samples_values), len(eps_values)))
heatmap_noise    = np.zeros((len(min_samples_values), len(eps_values)))

for i, minpts in enumerate(min_samples_values):
    for j, eps in enumerate(eps_values):
        lbls = DBSCAN(eps=eps, min_samples=minpts).fit_predict(X_sens)
        heatmap_clusters[i, j] = len(set(lbls)) - (1 if -1 in lbls else 0)
        heatmap_noise[i, j] = np.sum(lbls == -1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.heatmap(heatmap_clusters, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=[f'{e:.1f}' for e in eps_values],
            yticklabels=min_samples_values, ax=axes[0], linewidths=0.5)
axes[0].set_xlabel('eps'); axes[0].set_ylabel('min_samples')
axes[0].set_title('Número de Clusters Encontrados', fontweight='bold')

sns.heatmap(heatmap_noise, annot=True, fmt='.0f', cmap='Reds',
            xticklabels=[f'{e:.1f}' for e in eps_values],
            yticklabels=min_samples_values, ax=axes[1], linewidths=0.5)
axes[1].set_xlabel('eps'); axes[1].set_ylabel('min_samples')
axes[1].set_title('Número de Pontos de Ruído', fontweight='bold')

plt.suptitle('Sensibilidade dos Parâmetros do DBSCAN (Moons dataset)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🛒 7. Regras de Associação — Análise de Cesta de Compras

In [ ]:
# ================================================================
# Análise de Cesta de Compras com mlxtend
# ================================================================

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Simula base de compras de supermercado (100 transações, 15 produtos)
np.random.seed(SEED)
items_catalog = [
    'leite', 'pao', 'manteiga', 'queijo', 'frango',
    'arroz', 'feijao', 'macarrao', 'molho', 'cerveja',
    'agua', 'refrigerante', 'biscoito', 'chocolate', 'cafe'
]

# Probabilidades de cada item (frequências realistas)
item_probs = np.array([0.6, 0.55, 0.4, 0.35, 0.3,
                        0.5, 0.45, 0.3, 0.25, 0.2,
                        0.4, 0.3, 0.25, 0.2, 0.35])

# Adiciona correlações: leite+pao, arroz+feijao, macarrao+molho
transactions = []
for _ in range(150):
    basket = []
    for item, prob in zip(items_catalog, item_probs):
        if np.random.rand() < prob:
            basket.append(item)
    # Correlações artificiais
    if 'leite' in basket and np.random.rand() < 0.7:
        if 'pao' not in basket: basket.append('pao')
    if 'arroz' in basket and np.random.rand() < 0.8:
        if 'feijao' not in basket: basket.append('feijao')
    if 'macarrao' in basket and np.random.rand() < 0.75:
        if 'molho' not in basket: basket.append('molho')
    if len(basket) > 0:
        transactions.append(basket)

print(f'Total de transações: {len(transactions)}')
print(f'Exemplo: {transactions[0]}')

# Codifica em matriz binária
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_trans = pd.DataFrame(te_array, columns=te.columns_)

# Apriori
frequent_itemsets = apriori(df_trans, min_support=0.1, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
print(f'\nItemsets frequentes encontrados: {len(frequent_itemsets)}')

# Gera regras de associação
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5)
rules = rules.sort_values('lift', ascending=False)
print(f'Regras geradas: {len(rules)}')
print('\n--- Top 15 Regras (por lift) ---')
print(rules[['antecedents','consequents','support','confidence','lift']].head(15).to_string(index=False))

# Heatmap de suporte entre pares de itens
items_top = list(te.columns_)
support_matrix = pd.DataFrame(0.0, index=items_top, columns=items_top)
for item1 in items_top:
    for item2 in items_top:
        if item1 != item2:
            support_matrix.loc[item1, item2] = (df_trans[item1] & df_trans[item2]).mean()

plt.figure(figsize=(10, 8))
sns.heatmap(support_matrix, cmap='YlOrRd', linewidths=0.4,
            annot=True, fmt='.2f', cbar_kws={'label': 'Suporte Conjunto'})
plt.title('Heatmap de Suporte Entre Pares de Itens', fontweight='bold', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Visualização das Regras: Scatter Plot + Grafo de Rede
# ================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1) Scatter: support vs confidence, colorido por lift, tamanho por conviction
scatter = axes[0].scatter(
    rules['support'],
    rules['confidence'],
    c=rules['lift'],
    cmap='RdYlGn',
    s=rules['conviction'].clip(0, 5) * 30 + 10,
    alpha=0.75,
    edgecolors='white',
    linewidth=0.5
)
plt.colorbar(scatter, ax=axes[0], label='Lift')
axes[0].set_xlabel('Suporte', fontsize=11)
axes[0].set_ylabel('Confiança', fontsize=11)
axes[0].set_title('Regras de Associação\n(tamanho = conviction, cor = lift)', fontweight='bold')

# Anota top regras
for _, row in rules.head(5).iterrows():
    ant = ', '.join(list(row['antecedents']))
    cons = ', '.join(list(row['consequents']))
    axes[0].annotate(f'{ant}→{cons}', (row['support'], row['confidence']),
                     fontsize=6, xytext=(3, 3), textcoords='offset points', color='navy')

# 2) Grafo de rede com networkx (apenas regras com lift > 1.5)
if NX_AVAIL:
    G = nx.DiGraph()
    rules_strong = rules[rules['lift'] > 1.5].head(20)
    for _, row in rules_strong.iterrows():
        for ant in row['antecedents']:
            for cons in row['consequents']:
                G.add_edge(ant, cons, weight=row['lift'])

    pos = nx.spring_layout(G, seed=SEED, k=1.5)
    edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
    max_w = max(edge_weights) if edge_weights else 1
    widths = [w / max_w * 3 for w in edge_weights]

    nx.draw_networkx_nodes(G, pos, ax=axes[1], node_color=COLORS[0],
                            node_size=800, alpha=0.85)
    nx.draw_networkx_labels(G, pos, ax=axes[1], font_size=8, font_color='white')
    nx.draw_networkx_edges(G, pos, ax=axes[1], edge_color=COLORS[1],
                            width=widths, alpha=0.6, arrows=True,
                            arrowstyle='->', arrowsize=15, connectionstyle='arc3,rad=0.1')
    axes[1].set_title('Grafo de Regras de Associação\n(espessura = lift)', fontweight='bold')
    axes[1].axis('off')
else:
    axes[1].text(0.5, 0.5, 'networkx não disponível.\nInstale com: pip install networkx',
                 ha='center', va='center', fontsize=11)
    axes[1].axis('off')

plt.suptitle('Visualização de Regras de Associação', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🚨 8. Detecção de Anomalias

In [ ]:
# ================================================================
# Detecção de Anomalias — Múltiplos Métodos
# ================================================================

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.neighbors import LocalOutlierFactor

# Gera dataset sintético de detecção de anomalias
# Inliers: distribuição normal, outliers: pontos distantes
np.random.seed(SEED)
N_inliers = 400
N_outliers = 40

X_in  = np.random.randn(N_inliers, 2) * 1.5
X_out = np.random.uniform(-8, 8, (N_outliers, 2))
# Adiciona outliers distantes
X_out = np.vstack([X_out, np.array([[6, 6], [-6, -6], [5, -5], [-5, 5], [7, 0], [0, 7]])])

X_anom = np.vstack([X_in, X_out])
y_anom = np.hstack([np.zeros(N_inliers), np.ones(len(X_out))])  # 1=outlier

print(f'Dataset: {len(X_in)} inliers, {len(X_out)} outliers')
print(f'Taxa de contaminação: {len(X_out)/len(X_anom):.3f}')

contamination = len(X_out) / len(X_anom)
anom_results = []
anom_scores_dict = {}

# 1) Z-Score (estatístico)
zscores = np.abs(stats.zscore(X_anom)).max(axis=1)
preds_z = (zscores > 2.5).astype(int)
anom_results.append({'Método': 'Z-Score', 'AUC-ROC': roc_auc_score(y_anom, zscores),
                      'Acc': accuracy_score(y_anom, preds_z)})
anom_scores_dict['Z-Score'] = zscores

# 2) Isolation Forest (sklearn)
iforest = IsolationForest(contamination=contamination, random_state=SEED)
iforest.fit(X_anom)
scores_if = -iforest.score_samples(X_anom)  # negativo para que maior = mais anômalo
preds_if = (iforest.predict(X_anom) == -1).astype(int)
anom_results.append({'Método': 'Isolation Forest', 'AUC-ROC': roc_auc_score(y_anom, scores_if),
                      'Acc': accuracy_score(y_anom, preds_if)})
anom_scores_dict['Isolation Forest'] = scores_if

# 3) Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination)
preds_lof = (lof.fit_predict(X_anom) == -1).astype(int)
scores_lof = -lof.negative_outlier_factor_
anom_results.append({'Método': 'LOF', 'AUC-ROC': roc_auc_score(y_anom, scores_lof),
                      'Acc': accuracy_score(y_anom, preds_lof)})
anom_scores_dict['LOF'] = scores_lof

# 4) PyOD
if PYOD_AVAIL:
    for name, clf_od in [('IForest (PyOD)', IForest(contamination=contamination, random_state=SEED)),
                          ('LOF (PyOD)', LOF(contamination=contamination))]:
        clf_od.fit(X_anom)
        scores_od = clf_od.decision_scores_
        preds_od  = clf_od.labels_
        anom_results.append({'Método': name, 'AUC-ROC': roc_auc_score(y_anom, scores_od),
                              'Acc': accuracy_score(y_anom, preds_od)})
        anom_scores_dict[name] = scores_od

df_anom = pd.DataFrame(anom_results)
df_anom['AUC-ROC'] = df_anom['AUC-ROC'].round(4)
df_anom['Acc'] = df_anom['Acc'].round(4)
print('\n--- Comparação de Métodos de Detecção de Anomalias ---')
print(df_anom.to_string(index=False))

# Visualização
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Curvas ROC
for (name, scores), color in zip(anom_scores_dict.items(), COLORS):
    fpr, tpr, _ = roc_curve(y_anom, scores)
    auc = roc_auc_score(y_anom, scores)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Curvas ROC', fontweight='bold'); axes[0].legend(fontsize=7)

# Barras AUC-ROC
df_anom_sorted = df_anom.sort_values('AUC-ROC', ascending=True)
axes[1].barh(df_anom_sorted['Método'], df_anom_sorted['AUC-ROC'], color=COLORS[:len(df_anom)])
axes[1].set_xlabel('AUC-ROC'); axes[1].set_title('AUC-ROC por Método', fontweight='bold')
axes[1].set_xlim(0.5, 1.0)

# Scatter dos dados colorido por anomalia
axes[2].scatter(X_anom[y_anom == 0, 0], X_anom[y_anom == 0, 1],
                c=COLORS[0], s=15, alpha=0.5, label='Normal')
axes[2].scatter(X_anom[y_anom == 1, 0], X_anom[y_anom == 1, 1],
                c=COLORS[3], s=80, marker='x', linewidths=2, label='Anomalia')
axes[2].set_title('Dataset — Inliers vs Outliers', fontweight='bold')
axes[2].legend()

plt.suptitle('Detecção de Anomalias — Comparação de Métodos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🎓 9. Exercícios

**Exercício 1**: Calcular Silhouette manualmente  
**Exercício 2**: K-Means com diferentes K no dataset Wine  
**Exercício 3**: Apriori manual em base de transações  
**Exercício 4**: Comparação de DBSCAN e K-Means em dados de alta dimensão  
**Exercício 5**: Pipeline completo de clustering

In [ ]:
# ================================================================
# Templates de Exercícios
# ================================================================

# ---- Exercício 1: Silhouette Manual ----
# Dados: p1=(1,1), p2=(2,1), p3=(5,5), p4=(6,5)
# C1 = {p1, p2}, C2 = {p3, p4}

points = np.array([[1,1],[2,1],[5,5],[6,5]])
true_labels = np.array([0, 0, 1, 1])

# TODO: Calcule manualmente a(i) e b(i) para cada ponto
# a(i) = distância média intra-cluster do ponto i
# b(i) = distância média ao cluster vizinho mais próximo
# s(i) = (b(i) - a(i)) / max(a(i), b(i))

# Verifique com sklearn:
sil_check = silhouette_score(points, true_labels)
print(f'Exercício 1 — Silhouette (sklearn): {sil_check:.4f}')
print('(Calcule manualmente e verifique se coincide)\n')

# ---- Exercício 2: K-Means no Wine ----
wine = load_wine()
X_wine = StandardScaler().fit_transform(wine.data)
y_wine = wine.target

# TODO: Aplique K-Means com K=2,3,4,5
# Para cada K: inércia, silhouette, Davies-Bouldin, ARI
# O K ótimo coincide com o número real de classes (3)?

print('Exercício 2 — K ótimo no dataset Wine:')
for k in [2, 3, 4, 5]:
    km_w = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    lbls_w = km_w.fit_predict(X_wine)
    ari_w  = adjusted_rand_score(y_wine, lbls_w)
    sil_w  = silhouette_score(X_wine, lbls_w)
    print(f'  K={k}: Silhouette={sil_w:.3f} | ARI={ari_w:.3f} | Inércia={km_w.inertia_:.1f}')

# ---- Exercício 3: Pipeline Completo ----
# TODO: Implemente o pipeline:
# 1. Normalizar Wine
# 2. PCA para 2D
# 3. KMeans + HDBSCAN
# 4. Visualizar com cores por cluster
# 5. Calcular todas métricas

from sklearn.decomposition import PCA
X_wine_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_wine)

km_wine = KMeans(n_clusters=3, n_init=10, random_state=SEED)
lbls_km_wine = km_wine.fit_predict(X_wine_2d)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
for k, c in enumerate(COLORS[:3]):
    mask = lbls_km_wine == k
    plt.scatter(X_wine_2d[mask, 0], X_wine_2d[mask, 1], c=c, s=25, alpha=0.7, label=f'Cluster {k+1}')
plt.title('K-Means (K=3) — Wine Dataset (PCA 2D)', fontweight='bold')
plt.legend(fontsize=8)

plt.subplot(1, 2, 2)
for k, c in enumerate(COLORS[:3]):
    mask = y_wine == k
    plt.scatter(X_wine_2d[mask, 0], X_wine_2d[mask, 1], c=c, s=25, alpha=0.7, label=f'Classe {k+1}')
plt.title('Ground Truth — Wine Dataset (PCA 2D)', fontweight='bold')
plt.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'\nK-Means no Wine (ARI): {adjusted_rand_score(y_wine, lbls_km_wine):.4f}')
print('Exercícios concluídos!')